# Interactive Agent Session: Chapter2_LangChain_HR_RAG

This Jupyter notebook provides a sandbox to run AI Agent code securely block-by-block. Make sure you have your `.env` configured properly before executing the agent loop below.

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# 1. Load the business data/PDF
loader = PyPDFLoader("data/employee_handbook.pdf")
docs = loader.load_and_split()

# 2. Vectorize and store local database
vectorstore = Chroma.from_documents(documents=docs, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. Setup the Prompt Template
llm = ChatOpenAI(model="gpt-4-turbo")
system_prompt = (
    "You are an HR Assistant. Use the following retrieved context to answer the question.\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 4. Construct the RAG Chain
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 5. Execute
response = rag_chain.invoke({"input": "How many days of paid paternity leave am I entitled to?"})
print(response["answer"])
